# Technical Assignment (NLP / IR / Gen AI / Semantic Search)
**Objective:**  The purpose of this assignment is to assess your skills in Natural Language
 Processing (NLP), Information Retrieval (IR), Generative AI, and Semantic Search. You will
 need to source an open dataset, perform an in-depth analysis, build a model, and derive
 actionable insights.

 **Assignment Description:** You are tasked with sourcing an open dataset related to textual data
 and using it to address a problem of your choice within the realm of NLP, IR, Gen AI, or
 Semantic Search. The focus can be on tasks such as text classification, summarization, topic
 modeling, question answering, media recommendations, or semantic search.

## Table of Contents
 1. Dataset Sourcing
 2. Problem Definition
 3. Approach and Pipeline
 4. Implementation
 5. Evaluation
 6. Recommendations

### Dataset Sourcing
For this assignment I've selected the MIcrosoft News Dataset (MIND) to build a retrieval-augmented generation (RAG) system. It is a large-scale dataset of 160k English news articles and 15 million user behaviour logs that was curated specifically for recommendation tasks.

Main reasons behind the choice:
- Its rich textual content including title, abstract, body, category and entities, makes it suitable for NLP-related tasks, e.g. semantic search, information retrieval and question answering.
- It is not limited to one specific domain and covers a diversity of topics which allows to build and test a more generalizable system.
- It has user interaction data that can be used to evaluate retrieval quality.
- The use of news articles as entities seems to align well with Dow Jones' focus.

### Problem Definition
Problem: With the current volume of available content and news, finding and processing relevant information can be both extremely time-consuming and overwhelming. And even more so in case of time-sensitive material like news articles. This is a significant problem, given that 
 ○ Clearly define the problem you intend to solve using the chosen dataset.
 ○ Explain why this problem is significant and what impact solving it might have.

### Approach and Pipeline:
 ○ Outline your approach to solving the problem.
 ○ Describe the tools, libraries, and frameworks you plan to use.
 ○ Detail your pipeline, including data preprocessing, feature engineering, model
 building, and evaluation.

### Evaluation:
 ○ Evaluate the performance of your model(s) using appropriate metrics.
 ○ Discuss the strengths and limitations of your approach.

### Recommendations:
 ○ Offer recommendations for potential improvements or next steps.

### Implementation:
○ Implement your pipeline in a Jupyter notebook.
 ○ Ensure that your code is well-documented and includes explanations for each
 step.
 ○ Visualize and explain your results.

In [15]:
import os
import pandas as pd 
import numpy as np
import pinecone
import time
import torch
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import SpladeEncoder
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
import wordninja
import itertools
import json
from functools import reduce
import pickle

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#### Load and preprocess the data

In [17]:
interactions = pd.read_csv('data/train/behaviors.tsv', sep='\t')

In [18]:
news_df = pd.read_csv('data/train/news.tsv', sep='\t', header=None, names=['id', 'category', 'subcategory', 'news_title', 'news_abstract', 'url', 'entities_title', 'entities_abstract'])

In [19]:
news_specialized_df = news_df[news_df['category'].isin(['news', 'finance'])]

In [20]:
news_specialized_df.shape

(18881, 8)

#### Build the index

In [6]:
#instantiate Pinecone client
pc = pinecone.Pinecone(api_key=os.getenv('PINECONE_API'), pool_threads=6)

In [7]:
#create index if it doesn't exist
index_name = 'mind-small'
if index_name not in pc.list_indexes().names():
        print(f"Creating {index_name} index.")
        pc.create_index(
          name=index_name,
          metric="dotproduct",
          dimension=768,
          spec=pinecone.ServerlessSpec(
                                    cloud="aws",
                                    region="us-east-1"
                                  )
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)

In [8]:
#connect to index
index = pc.Index(index_name)
print(f"Connected to {index_name}.")

Connected to mind-small.


https://huggingface.co/FinLang/finance-embeddings-investopedia

In [5]:
dense_vector_model = SentenceTransformer('FinLang/finance-embeddings-investopedia', device=device)
sparse_vector_model = SpladeEncoder(max_seq_length=512, device=device)


BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [10]:
#using lightweight probabilistic segmentation with wordninja
news_specialized_df.loc[:, "subcategory"] = news_specialized_df.loc[:,"subcategory"].apply(lambda x: ' '.join(wordninja.split(x)))

In [13]:
def parse_and_filter_entities(entities_string):
    try:
        entities = json.loads(entities_string)
        labels = [item['Label'] for item in filter(lambda a: a['Confidence'] > 0.95, entities)]
    except:
        labels = []
    return labels

In [15]:
#parse entities fields as json and combine those for titles and abstracts in one field
news_specialized_df["entities"] = news_specialized_df[['entities_title', 'entities_abstract']].apply(lambda x: list(set(reduce(lambda a, b: a+b, list(map(parse_and_filter_entities, x))))), axis=1)

C:\Users\eremi\AppData\Local\Temp\ipykernel_23580\1500707131.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_specialized_df["entities"] = news_specialized_df[['entities_title', 'entities_abstract']].apply(lambda x: list(set(reduce(lambda a, b: a+b, list(map(parse_and_filter_entities, x))))), axis=1)


In [16]:
data_records = news_specialized_df.to_dict('records')
data = []
for record in data_records:
    data.append({
            "id": record["id"],
            "context": f""""{record["news_title"]}\n{record["news_abstract"]}""",
            "title": record["news_title"],
            "category": record["category"],
            "subcategory": record["subcategory"],
            "entities": record["entities"],
            "sources": record["url"],
        })

In [ ]:
#calculate dense and sparse embeddings and load them asyncroniously to the vector database together with associated metadata
# it = iter(data)
it = pickle.load(open('iter_pickle.pkl','br'))

In [10]:
chunk = tuple(itertools.islice(it, 128))
while chunk:
    ids = [str(x["id"]) for x in chunk]
    contexts = [x["context"] for x in chunk]
    sources = [x['sources'] for x in chunk]
    titles = [x["title"] for x in chunk]
    categories = [x["category"] for x in chunk]
    subcategories = [x["subcategory"] for x in chunk]
    entities = [x["entities"] for x in chunk]
    
    dense_vectors = dense_vector_model.encode(contexts).tolist()
    sparse_vectors = sparse_vector_model.encode_documents(contexts)
    upsert_chunk = []

    for _id, dense_vector, sparse_vector, context, source, title, category, subcategory, entity in zip(ids,
                                                            dense_vectors,
                                                            sparse_vectors,
                                                            contexts,
                                                            sources,
                                                            titles,
                                                            categories,
                                                            subcategories,
                                                            entities
                                                            ):
        metadata = {
            "category": category,
            "subcategory": subcategory,
            "title": title,
            "context": context,
            "entities": entity,
            "source": source 
        }
        upsert_chunk.append(
            {
                "id": _id,
                "values": dense_vector,
                "sparse_values": sparse_vector,
                "metadata": metadata
            }
    )
    async_result = index.upsert(vectors=upsert_chunk, async_req=True)
    chunk = tuple(itertools.islice(it, 128))
    pickle.dump(it, open('iter_pickle.pkl', 'bw'))

#### Candidates Retrieval and Generating Final Recommendations <a class="anchor" id="rag"></a>

In [21]:
def encode_scale_vectors(query, alpha):
        """Encode sparse and dense vectors, and apply hybrid vector scaling using a convex combination alpha * dense + (1 - alpha) * sparse
        Args:
        alpha (float): 0 == sparse only, 1 == dense only
        """
        #create sparse query vector
        sparse = sparse_vector_model.encode_queries(query)
        #create dense query vector
        dense = dense_vector_model.encode(query).tolist()
        #scale sparse and dense vectors
        hyb_sparse = {
            'indices': sparse['indices'],
            'values':  [v * (1 - alpha) for v in sparse['values']]
        }
        hyb_dense = [v * alpha for v in dense]
        return hyb_dense, hyb_sparse

In [22]:
def retrieve_from_pinecone(index, query, metadata_filter, top_k, alpha=0.5):
    hyb_dense, hyb_sparse = encode_scale_vectors(query=query, alpha=alpha)
    retrieved_docs = index.query(
        top_k=top_k,
        vector=hyb_dense,
        sparse_vector=hyb_sparse,
        include_metadata=True,
        include_values=False,
        filter=metadata_filter
    )
    documents = [Document(page_content=f"""News article: {item['metadata']['context']}\nTopic: {item['metadata']['subcategory']}\nKeywords{item['metadata']['entities']}""",
        metadata={"title": item['metadata']['title'],
        'source': item['metadata']['source']}) for item in retrieved_docs["matches"]]
    _ids = [item['id'] for item in retrieved_docs["matches"]]
    return documents, _ids

In [89]:
#analyse user query and generate retriever filter based on it

In [23]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Optional
from pydantic import BaseModel as PydanticBaseModel
from typing_extensions import Annotated, TypedDict
import pycountry
from pydantic import Field
from tenacity import(
    retry,
    stop_after_attempt,
    wait_random_exponential
)

In [32]:
class BaseModel(PydanticBaseModel):
    class Config:
        arbitrary_types_allowed = True

class QueryDetails(BaseModel):
    category: list = Field(description="List of categories of news article the user is looking for",
                          enum=subcategories)
    named_entities: list = Field(description="List of named entitites.")

In [33]:
class QARetriever():
    def __init__(self):
        self.llm = ChatOllama(model= 'llama3.2', temperature=0.0)
        self.query_structured_llm = self.llm.with_structured_output(QueryDetails)
        self.query_classification_prompt = ChatPromptTemplate.from_template(
        """
        Analyse user's query and extract th following properties:
        1. List of categories of the news the user is interested in. Choose all the best matches from this list: {subcategories}.
        2. Main named entities and keyword phrases from the user's input.
        Use only data mentioned directly in the input. Output only those properties that you are absolutely sure of based on the user's input.
        Input: {input}
        """
        )

        self.query_classification_chain = self.query_classification_prompt | self.query_structured_llm

    @retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(3))

    def infer_query_metadata(self, query):
        subcategories = news_specialized_df.subcategory.unique().tolist()
        return self.query_classification_chain.invoke({'subcategories': subcategories, 'input': query})

    def retrieve_docs(self, user_query):
        user_metadata = self.infer_query_metadata(query=user_query)
        user_category = user_metadata.category
        user_entities = user_metadata.named_entities

        metadata_filter = {
            "$and":
            
                    [
                        {"subcategory": {"$in": user_category}},
                        {"entitites": {"$in": user_entities}}]
        }

        metadata_light_filter = {
            "$or":
                    [
                        {"subcategory": {"$in": user_category}},
                        {"entitites": {"$in": user_entities}}]
        }

        #retrieve 3 batches of candidates with different filters
        retrieved_documents = []
        top_k = 20

        #with full filters applied
        retrieved_filtered_documents, _ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_filter, top_k=top_k, alpha=0.5)
        retrieved_documents.extend(retrieved_filtered_documents)

        #with or-filters applied, and don't repeat already retrieved matches
        metadata_light_filter["id"] = {"$nin": _ids}
        top_k = max(3, top_k - len(_ids))
        retrieved_or_filter_documents, add_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_light_filter, top_k=top_k, alpha=0.5)
        _ids.extend(add_ids)
        retrieved_documents.extend(retrieved_or_filter_documents)

        #without filter and don't repeat already retrieved matches
        top_k = max(2, top_k - len(add_ids))
        metadata_minimal_filter = {"id": {"$nin": _ids}}
        retrieved_minimal_filter_documents, last_ids = retrieve_from_pinecone(index=index, query=user_query, metadata_filter = metadata_minimal_filter, top_k=top_k, alpha=0.5)
        retrieved_documents.extend(retrieved_minimal_filter_documents)

        return retrieved_documents


Pinecone doesn´t allow should or boost

In [37]:
class Articles(BaseModel):
    title: str = Field(description="Title of this particular relevant article")
    source: str = Field(description="Source url of this particular relevant article")
    summary: str = Field(description="Short summary of this particular relevant article")


class RecOutput(BaseModel):
    articles: List[Articles] = Field(description="A list of recommended articles used for the answer.")
    reply: str = Field(description="A conversational output with an answer to the user's question.")

In [44]:
class QAGeneration():
    def __init__(self):
        self.rag_llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.2, frequency_penalty = 0.1, api_key=os.getenv('OPENAI_API_KEY'))

        self.system_prompt = (
        """You are an assistant for question-answering tasks and recommending news articles. Use the following pieces of retrieved context to answer the question. {context}. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
        Return two things:
        1. A conversational output where you give an anwer to the user's question in one short paragraph.
        2. A list of documents, that you used to answer the question. Each document should be represnted as a json with title, source and summary. Make sure that all elements in one json come from the same document.
        """
        )

        self.qa_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt),
                ("human", "{input}"),
            ]
        )

        self.rag_chain_from_docs = (
                    {
                        "input": lambda x: x["input"],
                        "context": lambda x: x["context"]
                    }
                    | self.qa_prompt
                    | self.rag_llm.with_structured_output(RecOutput)
                )
        self.chain = RunnablePassthrough.assign(answer=self.rag_chain_from_docs)
        
    def get_answer(self, user_query, retrieved_documents):
        rag_output = self.chain.invoke({"input": user_query, "context": retrieved_documents})
        conv_reply = rag_output['answer'].reply
        articles = rag_output['answer'].articles
        return conv_reply, articles

In [46]:
qa_retriever = QARetriever()
qa_generator = QAGeneration()

In [ ]:
user_query = input('Ask your question: ')

In [61]:
retrieved_docs = qa_retriever.retrieve_docs(user_query=user_query)

In [62]:
conv_reply, articles = qa_generator.get_answer(user_query=user_query, retrieved_documents=retrieved_docs)

In [63]:
print(conv_reply)

Here are some articles about Goldman Sachs: one discusses their recent insider trading charges, another covers disappointing banking revenue and writedowns, and the last one details their retreat from Ken Fisher's firm after controversial remarks.


In [64]:
for article in articles:
    print(article.title, '\n', article.summary, '\n', article.source, '\n\n')

Goldman's Unwelcome Streak: A String of Insider Trading Charges 
 Goldman Sachs Group Inc. has built itself into a global dealmaking force whose bankers are often stitching together the biggest and most sensitive corporate transactions. 
 https://assets.msn.com/labs/mind/AAJ2bsv.html 


Goldman Sachs' quarter soured by disappointing banking revenue, writedowns for Uber and other investments 
 Goldman took a $267 million markdown on public investments in companies like Uber and Tradeweb Markets. 
 https://assets.msn.com/labs/mind/AAIOfrq.html 


Goldman Sachs Joins Retreat From Fisher's Firm After His Remarks 
 Goldman Sachs Group Inc. is joining the exodus from Ken Fisher's firm. 
 https://assets.msn.com/labs/mind/AAJjDGx.html 


